Steps:

0. strip whitespace for the Abkürzung and Auflösung columns
1. make the inheriting of abbreviations explicit
    - The first column value is inherited from the last line that had a value in column 1.
2. transform so that each row only contains it's own abbreviation value in it's RG1 to RG9 columns
    1. this is only done for rows that only contain abbreviations or their own expansion in the RG columns. Rows where this doesn't hold, are often ones where it's unclear what the abbreviation actually stands for. For these, special handling will be necessary.
    2. information 
3. check whether abbreviations actually occur in the RG text
    1. keep only rows with abbreviations that do occur
    3. note abbreviations that don't occur exactly, but where something similar occurs
4. note in which volumes abbreviations occur

## step 0

In [ ]:
import polars as pl
pl.read_csv("data/abbreviations.csv").with_columns(
    pl.col("Abkürzung").str.strip_chars().alias("Abkürzung"),
    pl.col("Auflösung").str.strip_chars().alias("Auflösung"),
    ).write_csv("data/abbreviations_wo_whitespace.csv")

## step 1

In [1]:
import csv

input_file = 'data/abbreviations_wo_whitespace.csv'
output_file = 'data/explicit_abbreviations.csv'

last_value = ''

with open(input_file, 'r', encoding='utf-8') as infile, open(output_file, 'w', encoding='utf-8', newline='') as outfile:
    reader = csv.reader(infile)
    writer = csv.writer(outfile)
    
    # Write header as-is
    header = next(reader)
    writer.writerow(header)
    
    for row in reader:
        if row[0].strip():  # If first column has a value
            row[0] = row[0].strip() # remove whitespace
            last_value = row[0]
        else:  # Inherit from last non-empty value
            row[0] = last_value
        writer.writerow(row)

In [2]:
import polars as pl

explicit = pl.read_csv("data/explicit_abbreviations.csv")
explicit.sort(by=[pl.col("Abkürzung").str.to_lowercase(), pl.col("Auflösung")]).write_csv("data/explicit_abbreviations.csv")

## step 2

In [3]:
from process_abbreviations import process_csv

input_file = 'data/explicit_abbreviations.csv'
output_file = 'data/processed_abbreviations.csv'

process_csv(input_file, output_file)

sorting afterwards was simpler

In [4]:
processed = pl.read_csv("data/processed_abbreviations.csv")
processed.sort(by=[pl.col("Abkürzung").str.to_lowercase(), pl.col("Auflösung")]).write_csv("data/processed_abbreviations.csv")

## step 3

In [5]:
text = pl.read_csv("data/RG_header_sublemma_all.csv")
text = text.with_columns(
    pl.coalesce([pl.col("header_no_tags"), pl.col("sublemma_no_tags")]).alias("text")
    #pl.col("volume").cast(pl.String)
).select(["volume", "text"])

abbr = pl.read_csv("data/processed_abbreviations.csv")

In [6]:
def search(text, query, literal=False, show=True) -> tuple[bool, str]:
    """
    returns a tuple: (whether the query exists at all in the RG, in which volumes it exists)
    """
    result = text.filter(pl.col("text").str.contains(query, literal=literal)).group_by("volume").len().sort("volume")
    total = result.select(pl.col("len").sum()).item()
    if(show):
        display(result)
        print(f"Total: {total}")

    s = ""
    for x in result.get_column("volume").to_list():
        s+= str(x) + '|'
    s = s[:-1]
    
    return total > 0, s

In [7]:
abbr.filter(~pl.col("Abkürzung").str.ends_with("."))

Abkürzung,Auflösung,Übersetzung,Anmerkungen,Wortstamm,Deklination,RG1,RG2,RG3,RG4,RG5,RG6,RG7,RG8,RG9,Bemerkungen_1,Bemerkungen_2
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""def. nat. (c. c.)""","""defectus natalium (de coniugat…","""Geburtsmakel (von einem Verhei…",null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""def. nat. (c. s.)""","""defectus natalium (de coniugat…","""Geburtsmakel (von einem Verhei…",null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""def. nat. (p. c.)""","""defectus natalium (de presbite…","""Geburtsmakel (von einem Priest…",null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""def. nat. (p. s.)""","""defectus natalium (de presbite…","""Geburtsmakel (von einem Priest…",null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""def. nat. (s. c.)""","""defectus natalium (de soluto e…","""Geburtsmakel (von einem Ledige…",null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""def. nat. (s. s.)""","""defectus natalium (de soluto e…","""Geburtsmakel (von einem Ledige…",null,null,null,null,null,null,null,null,null,null,null,null,"""alle Kombinationen mit def. na…",null
"""hosp. pauperum""","""hospitale pauperum""",null,null,"""nur hospitale wird dekliniert""",null,"""hosp. pauperum""","""hosp. pauperum""","""hosp. pauperum""","""hosp. pauperum""",null,null,null,"""hosp. pauperum""","""hosp. pauperum""",null,null
"""o. s. Clare""","""ordo sancte Clare""","""Klarissenorden""",null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""Rom. curia""","""Romana curia""",null,"""1""","""Roman -(a)e curi -(a)e""","""beide a""","""Rom. curia; curia Rom.""","""Rom. curia; Rom.""","""Rom. curia; Roman. cur.; Rom. …","""Rom. curia; curia Rom.""",null,"""Rom. cur.; cur. Rom.""","""Romana cur.; cur. Romana""","""Romana cur.""","""Romana cur.; cur. Romana""",null,null


In [8]:
import re

def construct_query(abbreviation):
    escaped = re.escape(abbreviation)
    escaped = escaped.replace(r'\.', r'\.?')
    if escaped.endswith(r'\.?'):
        escaped = escaped[:-3] + r'[\., ]'
    return r'[^\w]' + escaped

In [9]:
abbr = abbr.with_columns(
    # `.` seems to sometimes be missing (or at least it's likely), so we don't match literally, but with `.` optionally missing
    #pl.col("Abkürzung").map_elements(lambda x: search(text, x, literal=True, show=False)[0]).alias("exists"),
    pl.col("Abkürzung").map_elements(lambda x: search(text, construct_query(x), literal=False, show=False)[0]).alias("exists"),
    pl.col("Abkürzung").map_elements(lambda x: search(text, re.escape(x[:-1]) + "[^a-zA-Z]", literal=False, show=False)[0]).alias("exists_similar"),
    pl.col("Abkürzung").map_elements(lambda x: search(text, "(" + "|".join(re.escape(x).split(r"\ ")) + ")", literal=False, show=False)[0], pl.Boolean).alias("part_exists")
)
abbr

Abkürzung,Auflösung,Übersetzung,Anmerkungen,Wortstamm,Deklination,RG1,RG2,RG3,RG4,RG5,RG6,RG7,RG8,RG9,Bemerkungen_1,Bemerkungen_2,exists,exists_similar,part_exists
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,bool,bool,bool
"""a.""","""?""",null,null,null,null,null,"""a.""",null,null,null,null,null,null,null,null,"""(Bd. II 25)""",true,true,true
"""a.""","""annus""","""Jahr""","""siehe Eintrag ""an.""""","""ann -i""","""o """,null,null,null,null,null,null,null,null,null,null,null,true,true,true
"""a.""","""annus""","""Jahr""",null,"""ann -i""","""o""","""a.""",null,"""a.""",null,null,null,null,null,null,null,null,true,true,true
"""a.""","""argentum""","""Silber""",null,"""argent -i""","""o (n.)""",null,null,null,null,null,"""a.""","""a.""",null,"""a.""",null,"""meist in Verbindung m. a. p.; …",true,true,true
"""abb.""","""abbas""","""Abt""",null,"""abbat -is""","""konsonantisch (m.)""","""abb.""","""abb.""","""abb.""","""abb.""","""abb.""","""abb.""","""abb.""","""abb.""","""abb.""",null,null,true,true,true
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""vig.""","""vigore""","""kraft""","""vigor in Bd. 6,8""","""vigor -is""","""konsonantisch""",null,null,null,null,"""vig.""","""vig.""","""vig.""","""vig.""","""vig.""",null,null,true,true,true
"""virg.""","""virgo""","""Jungfrau""","""in 4 auch v.""","""virgin -is""","""konsonantisch""","""virg.""","""virg.""",null,"""virg.""","""virg.""","""virg.""","""virg.""","""virg.""","""virg.""",null,null,true,true,true
"""visit.""","""visitare;""","""besuchen; kontrollieren;""",null,"""visit, visitav, visitat""","""a-Konj.""","""visit.""","""visit.""",null,"""visit.""","""visit.""","""visit.""","""visit.""","""visit.""","""visit.""",null,null,true,true,true


In [10]:
with pl.Config(tbl_rows=-1):
    display(abbr.filter((pl.col("exists") == False) & (pl.col("exists_similar") == True)).get_column("Abkürzung").unique().sort())

Abkürzung
str
"""Ap."""
"""camerar."""
"""confirm."""
"""dict."""
"""flor."""
"""imp."""
"""k."""
"""laci."""
"""obs."""


In [11]:
#with pl.Config(tbl_rows=-1):
    #display(abbr.filter((pl.col("part_exists") == True) & (pl.col("exists") == False)))
abbr.filter((pl.col("part_exists") == True) & (pl.col("exists") == False))

Abkürzung,Auflösung,Übersetzung,Anmerkungen,Wortstamm,Deklination,RG1,RG2,RG3,RG4,RG5,RG6,RG7,RG8,RG9,Bemerkungen_1,Bemerkungen_2,exists,exists_similar,part_exists
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,bool,bool,bool
"""ben. c. c.""","""beneficium cum cura""","""Pfründe mit Seelsorge""","""siehe benef. c. c.""",null,null,null,null,null,null,null,null,null,null,null,null,null,false,false,true
"""ben. c. c.""","""beneficium cum cura""","""Pfründe mit Seelsorge""",null,null,null,"""ben. c. c.""",null,null,null,null,null,null,null,null,null,null,false,false,true
"""ben. s. c.""","""beneficium sine cura""","""Pfründe ohne Seelsorge""","""siehe benef s. c.""",null,null,null,null,null,null,null,null,null,null,null,null,null,false,false,true
"""ben. s. c.""","""beneficium sine cura""","""Pfründe ohne Seelsorge""",null,null,null,"""ben. s. c.""",null,null,null,null,null,null,null,null,null,null,false,false,true
"""camerar.""","""camerarius""","""Kämmerer""",null,"""camerari -i""","""o""",null,"""camerar.""","""camerar.""","""camerar.""",null,null,null,null,null,null,null,false,true,true
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""testim. litt.""","""testimoniales littere""","""Bescheinigung, Zeugnis""",null,null,null,null,"""testim. litt.; litt. testim.; …",null,"""litt. testim.""","""testim. litt.; litt. testim.; …","""litt. testim.""","""litt. testim.""","""testim. litt.""","""litt. testim.""",null,null,false,false,true
"""thesaurar.""","""thesauraria""","""Amt des Thesaurars""",null,"""thesaurari -(a)e""","""a""","""thesaurar.""","""thesaurar.""","""thesaurar.""","""thesaurar.""",null,null,null,null,null,null,null,false,true,true
"""thesaurar.""","""thesauraria""","""Amt des Thesaurars """,null,null,null,null,null,null,null,null,null,null,null,null,null,null,false,true,true


In [12]:
abbr.filter(pl.col("exists") == False).get_column("Abkürzung").unique()

Abkürzung
str
"""penitent."""
"""protomart."""
"""sac."""
"""pecun."""
"""s. p. d."""
…
"""privileg."""
"""confirm."""
"""referend."""


In [13]:
abbr.filter(pl.col("exists") == True).drop("exists", "exists_similar", "part_exists").write_csv("data/abbreviations_with_matches.csv")

In [14]:
construct_query("thesaurar.")

'[^\\w]thesaurar[\\., ]'

In [15]:
abbr.filter(pl.col("Abkürzung").str.contains("thesaurar"))

Abkürzung,Auflösung,Übersetzung,Anmerkungen,Wortstamm,Deklination,RG1,RG2,RG3,RG4,RG5,RG6,RG7,RG8,RG9,Bemerkungen_1,Bemerkungen_2,exists,exists_similar,part_exists
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,bool,bool,bool
"""thesaurar.""","""thesauraria""","""Amt des Thesaurars""",null,"""thesaurari -(a)e""","""a""","""thesaurar.""","""thesaurar.""","""thesaurar.""","""thesaurar.""",null,null,null,null,null,null,null,false,true,true
"""thesaurar.""","""thesauraria""","""Amt des Thesaurars """,null,null,null,null,null,null,null,null,null,null,null,null,null,null,false,true,true
"""thesaurar.""","""thesaurarius""","""Amt des Thesaurars, Schatzmeis…",null,"""thesaurari -i""","""o""","""thesaurar.""",null,null,null,null,null,null,null,null,"""nicht auszuschließen, dass nic…",null,false,true,true
"""thesaurar.""","""thesaurarius""","""Amt des Thesaurars, Schatzmeis…","""siehe thes. und thesaur.""",null,null,null,null,null,null,null,null,null,null,null,null,null,false,true,true


## step 4

In [16]:
existing = pl.read_csv("data/abbreviations_with_matches.csv")
existing = existing.with_columns(
    #pl.col("Abkürzung").map_elements(lambda x: search(text, re.escape(x).replace(r'\.', r'\.?'), literal=False, show=False)[1]).alias("volumes")
    pl.col("Abkürzung").map_elements(lambda x: search(text, construct_query(x), literal=False, show=False)[1]).alias("volumes"),
)
existing

Abkürzung,Auflösung,Übersetzung,Anmerkungen,Wortstamm,Deklination,RG1,RG2,RG3,RG4,RG5,RG6,RG7,RG8,RG9,Bemerkungen_1,Bemerkungen_2,volumes
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""a.""","""?""",null,null,null,null,null,"""a.""",null,null,null,null,null,null,null,null,"""(Bd. II 25)""","""1|2|3|4|5|6|7|8|9|10"""
"""a.""","""annus""","""Jahr""","""siehe Eintrag ""an.""""","""ann -i""","""o """,null,null,null,null,null,null,null,null,null,null,null,"""1|2|3|4|5|6|7|8|9|10"""
"""a.""","""annus""","""Jahr""",null,"""ann -i""","""o""","""a.""",null,"""a.""",null,null,null,null,null,null,null,null,"""1|2|3|4|5|6|7|8|9|10"""
"""a.""","""argentum""","""Silber""",null,"""argent -i""","""o (n.)""",null,null,null,null,null,"""a.""","""a.""",null,"""a.""",null,"""meist in Verbindung m. a. p.; …","""1|2|3|4|5|6|7|8|9|10"""
"""abb.""","""abbas""","""Abt""",null,"""abbat -is""","""konsonantisch (m.)""","""abb.""","""abb.""","""abb.""","""abb.""","""abb.""","""abb.""","""abb.""","""abb.""","""abb.""",null,null,"""1|2|3|4|5|6|7|8|9|10"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""vig.""","""vigore""","""kraft""","""vigor in Bd. 6,8""","""vigor -is""","""konsonantisch""",null,null,null,null,"""vig.""","""vig.""","""vig.""","""vig.""","""vig.""",null,null,"""5|6|7|8|9|10"""
"""virg.""","""virgo""","""Jungfrau""","""in 4 auch v.""","""virgin -is""","""konsonantisch""","""virg.""","""virg.""",null,"""virg.""","""virg.""","""virg.""","""virg.""","""virg.""","""virg.""",null,null,"""1|2|4|5|6|7|8|9|10"""
"""visit.""","""visitare;""","""besuchen; kontrollieren;""",null,"""visit, visitav, visitat""","""a-Konj.""","""visit.""","""visit.""",null,"""visit.""","""visit.""","""visit.""","""visit.""","""visit.""","""visit.""",null,null,"""1|2|4|5|6|7|8|9|10"""


In [17]:
existing.filter(
    pl.any_horizontal(
        pl.col(f"RG{i}").is_not_null() & (~pl.col("volumes").str.contains(f"{i}"))
        for i in range(1, 10)
    )
)

Abkürzung,Auflösung,Übersetzung,Anmerkungen,Wortstamm,Deklination,RG1,RG2,RG3,RG4,RG5,RG6,RG7,RG8,RG9,Bemerkungen_1,Bemerkungen_2,volumes
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""abb. et. conv.""","""abbas et conventus""","""Abt und Konvent""",null,"""abbat -is; convent -us""","""konsonantisch/u""",null,"""abb. et. conv.""","""abb. et. conv.""","""abb. et. conv.""","""abb. et. conv.""","""abb. et. conv.""",null,"""abb. et. conv.""","""abb. et. conv.""",null,null,"""1|3|5|9|10"""
"""abol.""","""abolere;""","""tilgen; abschaffen;""",null,"""abole, abolev, abolit""","""e-Konj.""","""abolitio""",null,"""abol.""","""abolitio""","""abolitio""","""abol.""","""abol.""","""abol.""","""abol.; abolire""",null,null,"""3|6|7|8|9|10"""
"""acc.""","""accipere""","""empfangen, erhalten""",null,"""accipi, accep, accept""","""konsonantische Konj.""","""acceptare; accipere""","""acc.; accip.; acceptare; accip…","""acceptare; acceptatio; acciper…","""acc.; accept.; acceptare; acce…","""acc.; acceptare; acceptatio; a…","""acc.; acceptare; accipere""","""acc.; acceptare; acceptatio; a…","""acc.; acceptare; acceptatio; a…","""acc.; acceptare; acceptatio; a…",null,null,"""2|4|5|6|7|8|9|10"""
"""acol.""","""acolitus""","""Akoluth, Geistlicher mit der h…",null,"""acolit(h) -i; acolut -i""","""o""","""acolitus""","""acol.; acolit(h)us""","""acolit.; acolitus; acolutus""","""acol.; acolitus""","""acol.""","""acol.""","""acol.""","""acol.; acolitus""","""acol.; acolitus; acolutus""",null,null,"""2|4|5|6|7|8|9|10"""
"""adh.""","""adherens; adherentes""",null,null,"""adherent -is""","""gemischt""","""adh.; adher.; adherens; adhere…","""adher.; adherens; adherentes""","""adh.""","""adherens; adherentes""","""adherens; adherentes""","""adher.; adherens; adherentes""","""adherens; adherentes""","""adherens; adherentes""","""adherens; adherentes""",null,null,"""1"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""testim.""","""testimonialis""",null,"""in 4 auch test.""","""testimonial -is""","""i (Adj.)""",null,"""testim.""",null,"""testim.""","""testim.""","""testim.""","""testim.""","""testim.""","""testim.""","""möglicherweise auch in anderen…",null,"""9|10"""
"""thes.""","""thesaurarius""","""Amt des Thesaurars, Schatzmeis…",null,"""thesaurari -i""","""o""",null,"""thes.""",null,"""thes.""","""thes.""","""thes.""","""thes.""","""thes.""","""thes.""","""nicht auszuschließen, dass nic…",null,"""10"""
"""Tur.""","""Turonensis""","""Turnosen (Münze)""","""kommt ausgeschrieben nicht vor""",null,null,null,null,null,null,"""Tur.""",null,null,"""Tur.""",null,"""in Bd. 1,2,3 nicht verwendet, …",null,"""5"""


In [18]:
existing.write_csv("data/abbreviations_with_volumes.csv")

## distinguish between simple and more complex cases

In [19]:
existing = pl.read_csv("data/abbreviations_with_volumes.csv")
complex_abbreviations = existing.filter(pl.any_horizontal(~(pl.col("^RG[1-9]$").is_null() | pl.col("Abkürzung").str.contains(pl.col("^RG[1-9]$")))) | pl.any_horizontal(pl.col("^Abkürzung|Auflösung|RG[1-9]$").str.contains(";"))).get_column("Abkürzung").unique().to_list()
existing.filter(~pl.col("Abkürzung").is_in(complex_abbreviations)).write_csv("data/simple.csv")
existing.filter(pl.col("Abkürzung").is_in(complex_abbreviations)).write_csv("data/complex.csv")